In [1]:
# If needed, uncomment these in a notebook cell:
# !pip install ipywidgets scikit-learn

from ipywidgets import Textarea, Button, HTML, VBox, HBox, Layout, Output
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# --------------------------------------------------
# 1. Create a tiny "AI" model to classify skill level
# --------------------------------------------------

training_texts = [
    # Beginner style
    "I am new to Python",
    "I have never coded before",
    "I am a beginner at programming",
    "I just started learning Python",
    "I only know basic print statements",
    
    # Intermediate style
    "I know loops and functions",
    "I have used Python for data analysis",
    "I am comfortable with pandas and matplotlib",
    "I can write classes and handle files",
    "I know basic object oriented programming",
    
    # Advanced style
    "I build machine learning models in Python",
    "I work with deep learning and PyTorch",
    "I write production code and packages",
    "I optimize performance and use asyncio",
    "I contribute to open source Python projects"
]

training_labels = [
    "Beginner", "Beginner", "Beginner", "Beginner", "Beginner",
    "Intermediate", "Intermediate", "Intermediate", "Intermediate", "Intermediate",
    "Advanced", "Advanced", "Advanced", "Advanced", "Advanced"
]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(training_texts)

clf = LogisticRegression(max_iter=1000)
clf.fit(X, training_labels)

def predict_level(text):
    X_test = vectorizer.transform([text])
    return clf.predict(X_test)[0]

# --------------------------------------------------
# 2. Build the adaptive UI with ipywidgets
# --------------------------------------------------

# Input widgets
intro_html = HTML("<h3>Adaptive Python Tutor</h3><p>Describe your Python experience in your own words.</p>")
experience_text = Textarea(
    value="I have used Python a little for data analysis.",
    placeholder="Example: I have been using Python for a year for data analysis at work...",
    description="Your experience:",
    layout=Layout(width="100%", height="80px")
)

start_button = Button(
    description="Start tutorial",
    button_style="info",
    layout=Layout(width="200px")
)

result_html = HTML()
adaptive_area = VBox([])  # will hold the adaptive part of the UI
log_output = Output()

# --------------------------------------------------
# 3. Functions that build different UIs by level
# --------------------------------------------------

def build_beginner_ui():
    text = HTML("""
    <h4>Beginner mode</h4>
    <p>We will focus on the basics: variables, data types, and simple loops.</p>
    """)
    
    from ipywidgets import IntText, Button, Output
    num_widget = IntText(description="Number:", value=5)
    run_button = Button(description="Square it")
    out = Output()

    def on_run_clicked(b):
        with out:
            out.clear_output()
            print(f"{num_widget.value} squared is {num_widget.value ** 2}")

    run_button.on_click(on_run_clicked)
    return VBox([text, HBox([num_widget, run_button]), out])

def build_intermediate_ui():
    text = HTML("""
    <h4>Intermediate mode</h4>
    <p>We will focus on lists, list comprehensions, and basic data analysis style loops.</p>
    """)
    from ipywidgets import Text, Button, Output
    nums = Text(
        value="1,2,3,4,5",
        description="Numbers:"
    )
    run_button = Button(description="Compute squares")
    out = Output()

    def on_run_clicked(b):
        with out:
            out.clear_output()
            try:
                values = [int(x.strip()) for x in nums.value.split(",")]
                squares = [x * x for x in values]
                print("Input:", values)
                print("Squares:", squares)
            except Exception as e:
                print("Error parsing numbers:", e)

    run_button.on_click(on_run_clicked)
    return VBox([text, HBox([nums, run_button]), out])

def build_advanced_ui():
    text = HTML("""
    <h4>Advanced mode</h4>
    <p>We will focus on a tiny vectorized NumPy example, closer to data science workflows.</p>
    """)
    from ipywidgets import IntSlider, Button, Output
    import numpy as np

    n_slider = IntSlider(description="Array size", min=5, max=100000, step=5, value=1000)
    run_button = Button(description="Benchmark operation")
    out = Output()

    def on_run_clicked(b):
        with out:
            out.clear_output()
            n = n_slider.value
            x = np.random.randn(n)
            y = np.random.randn(n)
            result = (x * y).mean()
            print(f"Vector size: {n}")
            print(f"Mean of elementwise product: {result:.4f}")

    run_button.on_click(on_run_clicked)
    return VBox([text, HBox([n_slider, run_button]), out])

# --------------------------------------------------
# 4. Wire everything together
# --------------------------------------------------

def on_start_clicked(b):
    user_text = experience_text.value.strip()
    if not user_text:
        result_html.value = "<p style='color:red;'>Please describe your experience first.</p>"
        adaptive_area.children = []
        return
    
    # Use the "AI" model to predict skill level
    level = predict_level(user_text)
    result_html.value = f"<p>Detected level: <b>{level}</b></p>"
    
    if level == "Beginner":
        adaptive_area.children = [build_beginner_ui()]
    elif level == "Intermediate":
        adaptive_area.children = [build_intermediate_ui()]
    else:
        adaptive_area.children = [build_advanced_ui()]
    
    with log_output:
        print(f"User text: {user_text}")
        print(f"Predicted level: {level}")
        print("-" * 40)

start_button.on_click(on_start_clicked)

# Show the full UI
ui = VBox([
    intro_html,
    experience_text,
    start_button,
    result_html,
    adaptive_area,
    HTML("<hr><b>Debug log:</b>"),
    log_output
])

display(ui)
